# Full-fidelity ensemble release candidate

Two **execution profiles** for the exact same chronological experiment:

- **Day / Colab cloud**: leave your laptop available for work, train in a temporary Colab VM, and regularly back up a portable ZIP checkpoint to Google Drive
- **Overnight / local**: train on your laptop with up to eight XGBoost CPU threads, keeping the exact same training/selection/audit protocol
- Optional **quiet local**: two CPU threads while using your laptop

All use **700 trees**, the entire chronologically eligible history per fold, and full validation-month cohorts, NOT the 220-tree pilot. The initial budget is 2 generations × 3 candidates × 5 Optuna trials. Same hyperparameters, text bags, spatial features, seeds and member-selection rules regardless of execution profile.

**Stop/resume**: interrupt the factory and rerun the same cell to reuse completed work. Colab requires exporting/restoring the ZIP when its VM resets. The factory runs on local VM disk, never a live SQLite study on mounted Drive. Full commands are documented in [ENSEMBLE_RELEASE.md](ENSEMBLE_RELEASE.md).

**Be careful**: Free Colab does not guarantee unlimited runtime or background execution. The audit folds and historical six-month benchmark were consulted in earlier research: they are NOT brand-new untouched test sets. Don't enable the one-time audit until model selection is frozen.


## 1 · One-time Colab setup

In a Colab notebook run the following **only if running on Colab**, then clone or upload the project and ensure its working directory is the repository root. Install `optuna` plus the repository requirements. You need the **same housing data** and compatible Python packages locally and in Colab.

Do not create a new experiment directory when resuming: use the same relative `outputs/ensemble_release_candidate_full` path after restoring the ZIP.


In [4]:
# Only for Google Colab, after you've opened the repo in the VM.
MOUNT_GOOGLE_DRIVE = False
if MOUNT_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Install dependencies (if required):")
    print("%pip install -r requirements.txt optuna")


In [6]:
import subprocess
from pathlib import Path

repo = Path("/content/SSU-Data-Dash-hdnl-comp")

if not repo.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/HiddenL123/SSU-Data-Dash-hdnl-comp.git",
        str(repo)
    ], check=True)

In [5]:
%cd /content/SSU-Data-Dash-hdnl-comp
%pip install -r requirements.txt xgboost optuna

/content/SSU-Data-Dash-hdnl-comp
Ignoring numpy: markers 'python_version >= "3.14"' don't match your environment


In [6]:
import data

homes = data.load("listings")

print("Training homes:", homes.shape)
print("Columns:", len(homes.columns))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:140: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Training homes: (150000, 22)
Columns: 22


In [15]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

Path("/content/drive/MyDrive/automl").mkdir(
    parents=True, exist_ok=True
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2 · Choose where this session runs

`day_cloud` runs on Colab and requires a persistent external checkpoint ZIP.
`overnight_local` runs on the laptop at higher CPU utilization.
`day_local_quiet` runs locally with fewer threads. A profile switch does not change the experiment.

The same ZIP can be handed between the two machines, but **stop one worker before moving it**. A restore refuses nonempty destination folders, deliberately protecting the active experiment and Optuna database.


In [18]:
from pathlib import Path
import sys
import subprocess
import json
from IPython.display import display
import pandas as pd
from models.automl_profiles import get_profile, release_settings

RELEASE_PROFILE = "day_cloud" #"overnight_local"   # "day_cloud" or "day_local_quiet"
RELEASE_OUTPUT_DIR = "outputs/ensemble_release_candidate_full"
RELEASE_CHECKPOINT_ARCHIVE = (
    "/content/drive/MyDrive/automl/release_checkpoint.zip"
)     # e.g. /content/drive/MyDrive/automl/release_checkpoint.zip
RELEASE_GENERATIONS = 2
profile = get_profile(RELEASE_PROFILE)
settings = release_settings(output_dir=RELEASE_OUTPUT_DIR,
                            generations=RELEASE_GENERATIONS)
print("Execution:", profile)
print("Frozen model selection:", {k: settings[k] for k in (
    "trees", "train_cap", "validation_cap", "generations",
    "candidates_per_generation", "optuna_trials",
    "tuning_months", "selection_months", "audit_months"
)})


Execution: WorkerProfile(name='day_cloud', xgb_threads=2, description="Colab CPU runtime; free the user's laptop; use local VM disk and archive checkpoints to Drive", kind='cloud')
Frozen model selection: {'trees': 700, 'train_cap': 1000000, 'validation_cap': 1000000, 'generations': 2, 'candidates_per_generation': 3, 'optuna_trials': 5, 'tuning_months': ('2025-04', '2025-05'), 'selection_months': ('2025-06', '2025-07'), 'audit_months': ('2025-08', '2025-09', '2025-10')}


In [19]:
# Run or RESUME. Use CLI from a terminal for unattended local overnight jobs.
RUN_FULL_RELEASE_FACTORY = True
if RUN_FULL_RELEASE_FACTORY:
    cmd = [sys.executable, "ensemble_rc.py", "run",
           "--profile", RELEASE_PROFILE, "--output-dir", RELEASE_OUTPUT_DIR,
           "--generations", str(RELEASE_GENERATIONS)]
    if RELEASE_CHECKPOINT_ARCHIVE:
        cmd += ["--checkpoint-archive", RELEASE_CHECKPOINT_ARCHIVE]
    subprocess.run(cmd, check=True)


In [11]:
# Inspect latest durable state without training, tuning or opening audit.
path = Path(RELEASE_OUTPUT_DIR)
for name in ("progress.json", "roster.json", "summary.json"):
    item = path / name
    if item.is_file():
        obj = json.loads(item.read_text(encoding="utf-8"))
        if name == "roster.json":
            print("Processed:", len(obj["processed"]))
            print("Admitted:", [m["candidate"]["name"] for m in obj["members"]])
        elif name == "summary.json":
            print("Selection metrics:", obj.get("selection_metrics"))
            print("Weights:", obj.get("effective_member_weights"))
        else:
            print("Progress:", obj)


## 3 · Portable checkpoint handoff

Colab automatically backs up after completed training folds and candidate checkpoints if `RELEASE_CHECKPOINT_ARCHIVE` is set to a Google Drive path. To move work from Colab to your laptop: **stop the cloud process**, download `release_checkpoint.zip` and its accompanying `.sha256`, and restore into a NEW empty local output directory.

To move from overnight local to Colab: stop your local factory, export using the cell below, upload ZIP and checksum to Drive, and restore on Colab before starting a new cloud session.

**Important:** don't run both writers against one checkpoint. These are alternate *execution pathways*, not simultaneous distributed workers.


In [12]:
# Opt-in transfer operations. Never restore over existing work!
EXPORT_CHECKPOINT = False
RESTORE_CHECKPOINT = False
if EXPORT_CHECKPOINT:
    if not RELEASE_CHECKPOINT_ARCHIVE:
        raise ValueError("Specify an archive path, preferably on persistent storage")
    subprocess.run([sys.executable, "ensemble_rc.py", "export",
                    "--output-dir", RELEASE_OUTPUT_DIR,
                    "--checkpoint-archive", RELEASE_CHECKPOINT_ARCHIVE], check=True)
if RESTORE_CHECKPOINT:
    if not RELEASE_CHECKPOINT_ARCHIVE:
        raise ValueError("Specify path to a downloaded/Drive archive")
    subprocess.run([sys.executable, "ensemble_rc.py", "restore",
                    "--output-dir", RELEASE_OUTPUT_DIR,
                    "--checkpoint-archive", RELEASE_CHECKPOINT_ARCHIVE], check=True)


## 4 · Freeze and evaluate (one-time)

Complete all intended generations **before** opening the shadow audit. Once opened, the factory locks model membership and weights. Evaluate the frozen ensemble, then optionally run the **six-month December 2025–May 2026 historical benchmark** against the same full training-window protocol. That period has been studied before and is not a new independent test.

The full-history final refit follows only after the audit results are reviewed. An underperforming ensemble requires an explicit experimental override; no automatic claim of sub-7% MdAPE is made.


In [ ]:
RUN_FULL_RELEASE_AUDIT = False  # one-way roster lock
RUN_FROZEN_SIX_MONTH_BENCHMARK = False
if RUN_FULL_RELEASE_AUDIT or RUN_FROZEN_SIX_MONTH_BENCHMARK:
    commands = []
    if RUN_FULL_RELEASE_AUDIT:
        commands.append("audit")
    if RUN_FROZEN_SIX_MONTH_BENCHMARK:
        commands.append("benchmark")
    for action in commands:
        cmd = [sys.executable, "ensemble_rc.py", action,
               "--profile", RELEASE_PROFILE, "--output-dir", RELEASE_OUTPUT_DIR,
               "--generations", str(RELEASE_GENERATIONS)]
        if RELEASE_CHECKPOINT_ARCHIVE:
            cmd += ["--checkpoint-archive", RELEASE_CHECKPOINT_ARCHIVE]
        subprocess.run(cmd, check=True)


In [ ]:
# After frozen audit / benchmark review, save reusable full-history models.
FINALIZE_RELEASE_MODEL = False
EXPORT_OFFMARKET_SUBMISSION = False
ALLOW_EXPERIMENTAL_AUDIT_REGRESSION = False  # keep False for approved release
if FINALIZE_RELEASE_MODEL:
    cmd = [sys.executable, "ensemble_rc.py", "finalize",
           "--profile", RELEASE_PROFILE, "--output-dir", RELEASE_OUTPUT_DIR,
           "--generations", str(RELEASE_GENERATIONS)]
    if ALLOW_EXPERIMENTAL_AUDIT_REGRESSION:
        cmd.append("--allow-audit-regression")
    subprocess.run(cmd, check=True)
if EXPORT_OFFMARKET_SUBMISSION:
    subprocess.run([sys.executable, "ensemble_rc.py", "predict"], check=True)
    print("Written: submissions/submission_off_market_rc.csv")
